# Tutorial 20: Structured Backends — DuckDB, SQLite, Postgres, Mongo

Object stores work well for opaque blobs, but sometimes you want LAILA entries to live alongside structured data — a Postgres database that already holds your application state, a DuckDB file you can query analytically, a SQLite file for a small embedded app, or MongoDB for document storage.

You will:

- Round-trip a numpy array and a dict through each of the four backends
- See where each one keeps its data on disk (or in a managed local server)
- Note the trade-offs between embedded and server backends

**Prerequisites:** `pip install "laila-core[duckdb,postgres,mongo]"`. SQLite needs no extra (stdlib only).

In [ ]:
import numpy as np

import laila
from laila.pool import DuckDBPool, SQLitePool, PostgresPool, MongoPool

In [ ]:
def round_trip(pool, label):
    laila.memory.extend(pool, pool_nickname=label)
    arr = laila.constant(data=np.arange(16).reshape(4, 4), nickname=f"{label}_arr")
    cfg = laila.constant(data={"backend": label, "rows": 4}, nickname=f"{label}_cfg")
    laila.memorize(arr, pool_nickname=label).wait()
    laila.memorize(cfg, pool_nickname=label).wait()
    got_arr = laila.remember(nickname=f"{label}_arr", pool_nickname=label, persist=False).wait()
    got_cfg = laila.remember(nickname=f"{label}_cfg", pool_nickname=label, persist=False).wait()
    if isinstance(got_arr, list):
        got_arr = got_arr[0]
    if isinstance(got_cfg, list):
        got_cfg = got_cfg[0]
    print(f"  {label}: arr.shape={got_arr.data.shape}, cfg={got_cfg.data}")

## DuckDB — embedded analytical store

`DuckDBPool` writes to a `.duckdb` file. Without `file_path`, LAILA picks a default under the configured `pools/` directory.

In [ ]:
duck = DuckDBPool(nickname="duck")
round_trip(duck, "duck")

## SQLite — stdlib embedded store

`SQLitePool` needs no third-party extras — it uses the stdlib `sqlite3` module.

In [ ]:
sqlite = SQLitePool(nickname="sqlite")
round_trip(sqlite, "sqlite")

## PostgresPool — managed-local or remote

If you do **not** pass any connection parameters, `PostgresPool` starts a managed local `postgres` server in a subprocess. Pass `host` / `port` / `dbname` / `user` / `password` (or a `dsn`) to point at an existing server.

In [ ]:
try:
    pg = PostgresPool(nickname="pg")
    round_trip(pg, "pg")
except Exception as e:
    print("  postgres skipped:", type(e).__name__, "-", e)

## MongoPool — managed-local or remote

`MongoPool` mirrors the Postgres pattern: managed local `mongod` when no `uri`/`host` is given, otherwise connects to your existing cluster.

In [ ]:
try:
    mongo = MongoPool(nickname="mongo")
    round_trip(mongo, "mongo")
except Exception as e:
    print("  mongo skipped:", type(e).__name__, "-", e)

## When to pick a structured backend

| Backend | Reach for it when |
|---|---|
| DuckDB | You want to run analytical SQL across your entries on a single machine. |
| SQLite | You're shipping an embedded app and want a zero-dependency pool. |
| Postgres | You already run Postgres and want entries to live alongside business data. |
| Mongo | You already run MongoDB and prefer document semantics over rows. |

## Summary

- The `memorize` / `remember` API is unchanged across backends.
- DuckDB and SQLite are file-based; Postgres and Mongo can self-host or attach to existing servers.
- Pool routing (Tutorial 14) lets you mix structured and object backends in the same workflow.